# 04. Routing & Scenario Analysis (RUSH-1, RUSH-2, RUSH-3)

This notebook explores path planning using Dijkstra's algorithm across the scenarios.
- Enforcing Slide 85 diagonal corner-cutting constraints.
- Comparing Distance-Only vs. Traffic-Aware paths under objective cost $J(P)$.
- Investigating the RUSH-3 stress test and bottleneck relaxation.


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../code')
import starter_student as ss

train_df = pd.read_csv('../train_cells_capped.csv')
tr = train_df[train_df['map_id'] < 5]
model = ss.train_model(tr)

with open('../data/scenarios/scenarios.json') as f:
    scenarios = json.load(f)


In [ ]:
# Evaluate Scenario RUSH-1 (The Morning Commute)
sc1 = scenarios['RUSH-1']
sc1_df = pd.read_csv(f'../data/scenarios/{sc1["map"]}')
preds1 = ss.predict_scenario(model, sc1_df)
blocked1 = ss.blocking_policy(preds1, tau=0.18)
costs1 = ss.cell_cost(preds1)

start1, goal1 = sc1['start'], sc1['goal']
p_d1, l_d1, c_d1 = ss.plan_distance_only(blocked1, start1, goal1, costs1)
p_t1, l_t1, c_t1 = ss.plan_traffic_aware(blocked1, start1, goal1, costs1)

print('--- RUSH-1 Results ---')
print(f'Distance-Only: Length = {l_d1:.2f}, Cost = {c_d1:.2f}')
print(f'Traffic-Aware: Length = {l_t1:.2f}, Cost = {c_t1:.2f}')
print(f'Cost Reduction: {((c_d1 - c_t1)/c_d1)*100:.1f}% for +{((l_t1 - l_d1)/l_d1)*100:.1f}% distance')


In [ ]:
# RUSH-3 Stress Test Analysis: Investigating Connectivity
sc3 = scenarios['RUSH-3']
sc3_df = pd.read_csv(f'../data/scenarios/{sc3["map"]}')
preds3 = ss.predict_scenario(model, sc3_df)

for tau_test in [0.18, 0.25, 0.30, 0.40]:
    b_test = ss.blocking_policy(preds3, tau=tau_test)
    c_test = ss.cell_cost(preds3)
    p_d, l_d, c_d = ss.plan_distance_only(b_test, sc3['start'], sc3['goal'], c_test)
    p_t, l_t, c_t = ss.plan_traffic_aware(b_test, sc3['start'], sc3['goal'], c_test)
    found = 'Found' if p_t else 'Severed / Disconnected'
    print(f'Tau = {tau_test:.2f} -> Path Status: {found}')
    if p_t:
        print(f'   Traffic-Aware Cost: {c_t:.2f}, Distance: {l_t:.2f}')
